# Phase 2 — Flight Delay Prediction with K-Nearest Neighbors
Reconstructed from Pranjali’s slides 19–24. It reproduces the workflow (cleaning, encoding, imputation, scaling, tuning K=1…20), not the exact historical numerical results.

**Target definition:** delayed = 1 if either actual arrival or departure delay > 0. These outcome columns are then removed from model features to avoid direct target leakage. This is an *exploratory retrospective classification* rather than a pre-departure production prediction system.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
DATA = Path("../data/combined_flight_data.csv")
if not DATA.exists(): raise FileNotFoundError(f"Put the flight CSV at {DATA}")
df = pd.read_csv(DATA, na_values=["\\N", "\N"], low_memory=False)
df.columns = df.columns.str.strip().str.lower()
print(df.shape); display(df.head())

In [ ]:
if "delayed" not in df:
    if not {"arrdelay", "depdelay"}.issubset(df): raise ValueError("Need delayed label or arrdelay and depdelay")
    for c in ["arrdelay", "depdelay"]: df[c] = pd.to_numeric(df[c], errors="coerce")
    valid = df[["arrdelay", "depdelay"]].notna().any(axis=1)
    df = df.loc[valid].copy()
    df["delayed"] = ((df["arrdelay"].fillna(0)>0)|(df["depdelay"].fillna(0)>0)).astype(int)
else:
    df = df.loc[df["delayed"].notna()].copy()
    df["delayed"] = df["delayed"].replace({"Y":1,"N":0,"y":1,"n":0}).astype(int)
# Drop post-flight information and fields unsuitable for preflight predictions.
leakage = ["arrdelay", "depdelay", "arrtime", "actualelapsedtime", "airtime", "taxiin", "taxiout", "cancelled", "cancellationcode", "diverted", "tailnum"]
X = df.drop(columns=["delayed"] + leakage, errors="ignore")
y = df["delayed"]
# Avoid identifiers/high-cardinality strings dominating KNN.
X = X.drop(columns=[c for c in X if X[c].dtype == "object" and X[c].nunique(dropna=True)>150], errors="ignore")
for c in X.columns:
    if X[c].dtype == "object": continue
    X[c] = pd.to_numeric(X[c], errors="coerce")
X = X.dropna(axis=1, how="all")
print("Features:", list(X.columns)); print("Class distribution:", y.value_counts().to_dict())

In [ ]:
cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = X.columns.difference(cat_cols).tolist()
preprocess = ColumnTransformer([
    ("numeric", Pipeline([("impute", SimpleImputer(strategy="median")),("scale",StandardScaler())]),num_cols),
    ("categorical",Pipeline([("impute",SimpleImputer(strategy="most_frequent")),("onehot",OneHotEncoder(handle_unknown="ignore"))]),cat_cols)
])
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)
pipe = Pipeline([("prep",preprocess),("knn",KNeighborsClassifier())])
search = GridSearchCV(pipe,{"knn__n_neighbors":list(range(1,21))},cv=StratifiedKFold(3,shuffle=True,random_state=42),scoring="accuracy",n_jobs=-1)
search.fit(X_train,y_train)
print("Best K:",search.best_params_, "CV accuracy:",search.best_score_)
pred = search.predict(X_test)
print("Held-out accuracy:",accuracy_score(y_test,pred));print(classification_report(y_test,pred))

In [ ]:
results = pd.DataFrame(search.cv_results_)
plt.figure(figsize=(8,4))
plt.plot(range(1,21),results["mean_test_score"],marker="o")
plt.xlabel("Number of neighbors (K)");plt.ylabel("Cross-validation accuracy");plt.title("KNN accuracy vs K");plt.grid(True);plt.show()
ConfusionMatrixDisplay.from_predictions(y_test,pred);plt.show()

**Reproducibility:** The presentation suggests K=19 reached slightly above 0.77 accuracy, but this notebook intentionally calculates new metrics from the supplied dataset rather than hard-coding that number. KNN can be computationally expensive on very large datasets; consider a representative sample for local testing.